# BS-Detector validation and hyperparameter tuning

This notebook is designed for repeated validation after detector changes. It caches Cellpose segmentation labels once, then reruns the inexpensive detector/tracking stage for many parameter sets.

Workflow:
1. Point the notebook at an HDF5 movie and a completed `ground_truth_template.csv`.
2. Cache segmentation labels to Google Drive.
3. Run one configuration and inspect detector-vs-ground-truth overlays.
4. Sweep curvature-window, cap-exclusion, width, angle, and endpoint-offset parameters.
5. Save the best configuration, enriched detector CSV, metrics, and example plots.

Ground-truth rows are matched by `(video_id, frame, cell_name)`, following `validation/compare_to_ground_truth.py`.

## 1. Install and load the repository

In [ ]:
!pip install -q h5py scikit-image numpy scipy matplotlib pandas cellpose tifffile

import os, sys, json, csv, ast, itertools, pathlib, shutil
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = 'https://github.com/widenerm/pombe-bs-detector.git'
REPO_DIR = '/content/pombe-bs-detector'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Drive not mounted; local paths will be used.')

from pombe_tracker.config import Config
from pombe_tracker.io_utils import load_h5_data
from pombe_tracker.pipeline import CellProcessor, process_frame, _apply_lineage_poles
from pombe_tracker.segmentation import CellposeSegmenter
from pombe_tracker.tracking import CellTracker
from pombe_tracker.postprocessing import stabilize_scars
from validation.compare_to_ground_truth import compare, summarize, load_ground_truth_csv
print('Repository loaded:', REPO_DIR)

## 2. Configure the validation dataset

Set `H5_PATH`, `GROUND_TRUTH_PATH`, and `VIDEO_ID`. The ground-truth CSV can contain multiple videos; this notebook filters evaluation to `VIDEO_ID`.

In [ ]:
# ---- EDIT THESE VALUES ----
H5_PATH = '/content/drive/MyDrive/pombe_validation/your_movie.h5'
GROUND_TRUTH_PATH = '/content/drive/MyDrive/pombe_validation/ground_truth.csv'
VIDEO_ID = 'your_video_id'
DATASET_KEY = 'frames'
FRAME_LIMIT = None          # set an integer while debugging

# Keep this on Drive so segmentation is reusable across Colab sessions.
CACHE_DIR = '/content/drive/MyDrive/pombe_validation/bs_detector_cache'
OUTPUT_DIR = '/content/drive/MyDrive/pombe_validation/tuning_runs'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg = Config()
cfg.H5_FILE_PATH = H5_PATH
cfg.H5_DATASET_KEY = DATASET_KEY
cfg.NUM_FRAMES = FRAME_LIMIT
with h5py.File(H5_PATH, 'r') as h5:
    frames = h5[DATASET_KEY][:]
    frame_numbers = h5['frame_numbers'][:] if 'frame_numbers' in h5 else np.arange(len(frames))
if FRAME_LIMIT is not None:
    frames, frame_numbers = frames[:FRAME_LIMIT], frame_numbers[:FRAME_LIMIT]
frame_to_local = {int(number): i for i, number in enumerate(frame_numbers)}
gt_rows = [r for r in load_ground_truth_csv(GROUND_TRUTH_PATH)
           if r.get('video_id', '') == VIDEO_ID and int(r['frame']) in frame_to_local]
print(f'Frames: {len(frames)} | Ground-truth rows: {len(gt_rows)}')
display(pd.DataFrame(gt_rows).head())

## 3. Cache Cellpose labels

Run this once per movie or whenever segmentation settings/model change. Existing `.npy` labels are reused automatically.

In [ ]:
def cache_labels(frames, cfg, cache_dir, force=False):
    paths = [os.path.join(cache_dir, f'labels_{i:05d}.npy') for i in range(len(frames))]
    missing = [i for i, p in enumerate(paths) if force or not os.path.exists(p)]
    if missing:
        print(f'Running Cellpose for {len(missing)} frame(s)...')
        segmenter = CellposeSegmenter(cfg)
        for i in missing:
            labels = segmenter.segment(frames[i])
            np.save(paths[i], labels.astype(np.int32), allow_pickle=False)
            print(f'  cached frame {i}: {labels.max()} labels')
    else:
        print('All labels already cached.')
    return paths

label_paths = cache_labels(frames, cfg, CACHE_DIR, force=False)

class CachedSegmenter:
    def __init__(self, paths): self.paths = paths
    def segment(self, image, frame_index=None):
        if frame_index is None:
            raise ValueError('CachedSegmenter.segment requires frame_index')
        return np.load(self.paths[frame_index], allow_pickle=False)

# process_frame calls segment(image) without an index, so use a small
# frame-aware wrapper while keeping the pipeline code unchanged.
class SequentialCachedSegmenter:
    def __init__(self, paths): self.paths, self.i = paths, 0
    def reset(self): self.i = 0
    def segment(self, image):
        labels = np.load(self.paths[self.i], allow_pickle=False)
        self.i += 1
        return labels


## 4. Run one detector configuration

In [ ]:
def run_config(overrides, stabilize=True):
    run_cfg = Config()
    for key, value in overrides.items():
        setattr(run_cfg, key, value)
    run_cfg.NUM_FRAMES = len(frames)
    segmenter = SequentialCachedSegmenter(label_paths)
    processor = CellProcessor(run_cfg)
    tracker = CellTracker(run_cfg)
    all_results = []
    for frame_idx, frame in enumerate(frames):
        results = process_frame(frame, segmenter, processor)
        name_map = tracker.update(results, frame_idx=frame_idx)
        for r in results: r['cell_name'] = name_map.get(r['label'], '?')
        _apply_lineage_poles(results, tracker, frame_idx)
        all_results.append({'frame_idx': frame_idx, 'frame_number': int(frame_numbers[frame_idx]), 'frame': frame,
                            'cells': results, 'tracker': tracker})
    if stabilize:
        all_results, _ = stabilize_scars(all_results, run_cfg)
    return all_results, run_cfg

def point_string(point):
    return '' if point is None else str([float(x) for x in np.asarray(point)])

def detector_rows(all_results):
    rows = []
    for fd in all_results:
        for r in fd['cells']:
            rows.append({
                'video_id': VIDEO_ID, 'frame': fd['frame_number'],
                'cell_name': r.get('cell_name', '?'), 'area': r.get('area'),
                'scar_detected': r.get('scar_detected', False),
                'scar_midpoint': point_string(r.get('scar_midpoint')),
                'new_pole_point': point_string(r.get('new_pole_point')),
                'old_pole_point': point_string(r.get('old_pole_point')),
                'new_end_length': r.get('new_end_length'),
                'old_end_length': r.get('old_end_length'),
            })
    return rows

def evaluate(all_results):
    det = detector_rows(all_results)
    det_map = {(r['video_id'], int(r['frame']), r['cell_name']): r for r in det}
    joined, unmatched = compare(det_map, gt_rows)
    gt_positive = 0; tp = 0; fn = 0; fp = 0
    for gt in gt_rows:
        key = (VIDEO_ID, int(gt['frame']), gt['cell_name'])
        row = det_map.get(key)
        has_gt = bool(gt.get('gt_scar_mid_x_px', '').strip() and gt.get('gt_scar_mid_y_px', '').strip())
        has_det = bool(row and row.get('scar_detected'))
        gt_positive += has_gt
        tp += has_gt and has_det; fn += has_gt and not has_det; fp += (not has_gt) and has_det
    scar_dists = [r['scar_dist_px'] for r in joined if 'scar_dist_px' in r]
    return {
        'rows': len(gt_rows), 'matched': len(joined), 'unmatched': len(unmatched),
        'tp': int(tp), 'fn': int(fn), 'fp': int(fp),
        'recall': float(tp / max(tp + fn, 1)),
        'precision': float(tp / max(tp + fp, 1)),
        'scar_mae_px': float(np.mean(scar_dists)) if scar_dists else np.nan,
        'scar_median_px': float(np.median(scar_dists)) if scar_dists else np.nan,
    }

BASELINE = {
    'SCAR_CURVATURE_WINDOW': 0.08,
    'SCAR_CAP_EXCLUSION': 0.12,
    'SCAR_MAX_LONGITUDINAL_OFFSET': 0.08,
    'MIN_SCAR_WIDTH_RATIO': 0.80,
    'MAX_ANGLE_DEVIATION': 30.0,
}
results, run_cfg = run_config(BASELINE)
baseline_metrics = evaluate(results)
display(pd.Series(baseline_metrics))

## 5. Inspect detector-versus-ground-truth examples

In [ ]:
def parse_point(value):
    if not value: return None
    return np.asarray(ast.literal_eval(value), dtype=float)

def plot_examples(all_results, n=12):
    det = {(VIDEO_ID, fd['frame_number'], r['cell_name']): r
           for fd in all_results for r in fd['cells']}
    gt_map = {(VIDEO_ID, int(g['frame']), g['cell_name']): g for g in gt_rows}
    keys = list(gt_map)[:n]
    fig, axes = plt.subplots((len(keys)+3)//4, 4, figsize=(16, 4*((len(keys)+3)//4)))
    axes = np.atleast_1d(axes).ravel()
    for ax, key in zip(axes, keys):
        frame, name = key[1], key[2]; image = frames[frame_to_local[frame]]
        ax.imshow(image, cmap='gray')
        r = det.get(key); g = gt_map[key]
        if r is not None and r.get('contour') is not None:
            c = np.asarray(r['contour']); ax.plot(c[:,1], c[:,0], color='cyan', lw=.6)
            if r.get('scar_midpoint') is not None:
                p=np.asarray(r['scar_midpoint']); ax.scatter(p[1],p[0],c='lime',s=30,label='det')
        if g.get('gt_scar_mid_x_px','').strip() and g.get('gt_scar_mid_y_px','').strip():
            ax.scatter(float(g['gt_scar_mid_x_px']),float(g['gt_scar_mid_y_px']),c='red',s=25,label='GT')
        ax.set_title(f'f{frame} {name}'); ax.axis('off')
    for ax in axes[len(keys):]: ax.axis('off')
    plt.tight_layout(); plt.show()

plot_examples(results)

## 6. Hyperparameter sweep

Start with a small grid. Because segmentation is cached, this reruns only detector/tracking. Expand one parameter family at a time so the result is interpretable.

In [ ]:
SWEEP = {
    'SCAR_CURVATURE_WINDOW': [0.05, 0.08, 0.12],
    'SCAR_CAP_EXCLUSION': [0.08, 0.12, 0.16],
    'SCAR_MAX_LONGITUDINAL_OFFSET': [0.05, 0.08],
    'MIN_SCAR_WIDTH_RATIO': [0.70, 0.80, 0.90],
    'MAX_ANGLE_DEVIATION': [20.0, 30.0, 40.0],
}

# Choose a focused sweep first; the full Cartesian product can be large.
ACTIVE_SWEEP = ['SCAR_CURVATURE_WINDOW', 'SCAR_CAP_EXCLUSION']
grid = []
for values in itertools.product(*(SWEEP[k] for k in ACTIVE_SWEEP)):
    trial = dict(BASELINE); trial.update(dict(zip(ACTIVE_SWEEP, values))); grid.append(trial)

records = []
for i, trial in enumerate(grid, 1):
    trial_results, _ = run_config(trial)
    metrics = evaluate(trial_results)
    records.append({**trial, **metrics})
    print(f"{i:>3}/{len(grid)}  recall={metrics['recall']:.3f}  precision={metrics['precision']:.3f}  median={metrics['scar_median_px']:.2f}px")

sweep_df = pd.DataFrame(records)
sweep_df['objective'] = (sweep_df['recall'] * sweep_df['precision'] *
                         np.exp(-sweep_df['scar_median_px'].fillna(999) / 30.0))
sweep_df = sweep_df.sort_values('objective', ascending=False)
display(sweep_df.head(20))
sweep_df.to_csv(os.path.join(OUTPUT_DIR, 'hyperparameter_sweep.csv'), index=False)

## 7. Save and export the selected configuration

In [ ]:
BEST = sweep_df.iloc[0].to_dict() if len(sweep_df) else BASELINE
BEST_CONFIG = {k: BEST.get(k, v) for k, v in BASELINE.items()}
final_results, final_cfg = run_config(BEST_CONFIG)
final_metrics = evaluate(final_results)
run_name = 'selected_config'
with open(os.path.join(OUTPUT_DIR, run_name + '_config.json'), 'w') as f:
    json.dump(BEST_CONFIG, f, indent=2)
pd.DataFrame(detector_rows(final_results)).to_csv(
    os.path.join(OUTPUT_DIR, run_name + '_detector_rows.csv'), index=False)
with open(os.path.join(OUTPUT_DIR, run_name + '_metrics.json'), 'w') as f:
    json.dump(final_metrics, f, indent=2, allow_nan=True)
print('Selected configuration:', BEST_CONFIG)
display(pd.Series(final_metrics))
plot_examples(final_results)